In [11]:
import pickle
import pandas as pd
import numpy as np

# Настройка отображения
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

In [12]:
# Загрузка данных для всех типов швов
suture_types = ['COOS', 'ILS', 'IOVS_external', 'IOVS_internal']
all_data = {}

print("=== ЗАГРУЗКА ДАННЫХ ===")
for suture_type in suture_types:
    try:
        file_path = f'code and results/{suture_type}/architecture_results_cv.pkl'
        with open(file_path, 'rb') as f:
            all_data[suture_type] = pickle.load(f)
        print(f"✅ {suture_type}: загружено {len(all_data[suture_type])} архитектур")
    except FileNotFoundError:
        print(f"❌ {suture_type}: файл не найден")
        all_data[suture_type] = {}

print(f"\nВсего загружено данных для {len([k for k, v in all_data.items() if v])} типов швов")

=== ЗАГРУЗКА ДАННЫХ ===
✅ COOS: загружено 8 архитектур
✅ ILS: загружено 8 архитектур
✅ IOVS_external: загружено 8 архитектур
✅ IOVS_internal: загружено 8 архитектур

Всего загружено данных для 4 типов швов


In [13]:
# Создание детального датафрейма с результатами кросс-валидации для всех типов швов
print("=== СОЗДАНИЕ ДЕТАЛЬНОГО ДАТАФРЕЙМА ===")

all_cv_results = []

for suture_type, data in all_data.items():
    if not data:  # Пропускаем если данные не загружены
        continue
        
    print(f"\nОбработка {suture_type}...")
    
    for architecture, folds in data.items():
        for fold_idx, fold_data in enumerate(folds):
            # Основные метрики
            row = {
                'suture_type': suture_type,
                'architecture': architecture,
                'fold': fold_idx + 1,
                'accuracy': fold_data['accuracy'],
                'precision': fold_data['precision'],
                'recall': fold_data['recall'],
                'f1': fold_data['f1'],
                'auc': fold_data['auc'],
                'optimal_threshold': fold_data['optimal_threshold']
            }
            
            # Добавляем статистики по истории обучения
            history = fold_data['history']
            row.update({
                'total_epochs': len(history['accuracy']),
                'final_train_accuracy': history['accuracy'][-1],
                'final_train_loss': history['loss'][-1],
                'final_val_accuracy': history['val_accuracy'][-1] if history['val_accuracy'] else None,
                'final_val_loss': history['val_loss'][-1] if history['val_loss'] else None,
                'best_train_accuracy': max(history['accuracy']),
                'best_val_accuracy': max(history['val_accuracy']) if history['val_accuracy'] else None,
                'min_train_loss': min(history['loss']),
                'min_val_loss': min(history['val_loss']) if history['val_loss'] else None
            })
            
            all_cv_results.append(row)

# Создаем общий детальный DataFrame
df_cv_detailed = pd.DataFrame(all_cv_results)
print(f"\nСоздан общий детальный DataFrame с формой: {df_cv_detailed.shape}")
print(f"Столбцы: {list(df_cv_detailed.columns)}")

# Показываем количество записей по типам швов
print(f"\nКоличество записей по типам швов:")
suture_counts = df_cv_detailed['suture_type'].value_counts()
for suture, count in suture_counts.items():
    print(f"  {suture}: {count} записей")

print(f"\n=== ДЕТАЛЬНАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ (первые 10 строк) ===")
df_cv_detailed.head(10)


=== СОЗДАНИЕ ДЕТАЛЬНОГО ДАТАФРЕЙМА ===

Обработка COOS...

Обработка ILS...

Обработка IOVS_external...

Обработка IOVS_internal...

Создан общий детальный DataFrame с формой: (160, 18)
Столбцы: ['suture_type', 'architecture', 'fold', 'accuracy', 'precision', 'recall', 'f1', 'auc', 'optimal_threshold', 'total_epochs', 'final_train_accuracy', 'final_train_loss', 'final_val_accuracy', 'final_val_loss', 'best_train_accuracy', 'best_val_accuracy', 'min_train_loss', 'min_val_loss']

Количество записей по типам швов:
  COOS: 40 записей
  ILS: 40 записей
  IOVS_external: 40 записей
  IOVS_internal: 40 записей

=== ДЕТАЛЬНАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ (первые 10 строк) ===


,suture_type,architecture,fold,accuracy,precision,recall,f1,auc,optimal_threshold,total_epochs,final_train_accuracy,final_train_loss,final_val_accuracy,final_val_loss,best_train_accuracy,best_val_accuracy,min_train_loss,min_val_loss
0,COOS,EfficientNetB0,1,0.904762,1.000000,0.714286,0.833333,0.857143,0.43,56,0.785714,0.670166,0.761905,0.762346,0.833333,0.857143,0.670166,0.749675
1,COOS,EfficientNetB0,2,0.809524,0.625000,0.833333,0.714286,0.811111,0.37,55,0.797619,0.755374,0.761905,0.894019,0.809524,0.857143,0.748733,0.806032
2,COOS,EfficientNetB0,3,0.857143,0.800000,0.666667,0.727273,0.855556,0.32,55,0.809524,0.842084,0.666667,0.886775,0.821429,0.809524,0.793209,0.786711
3,COOS,EfficientNetB0,4,0.857143,0.800000,0.666667,0.727273,0.866667,0.47,50,0.750000,0.839310,0.714286,0.874086,0.797619,0.857143,0.793157,0.809148
4,COOS,EfficientNetB0,5,0.857143,1.000000,0.500000,0.666667,0.722222,0.25,55,0.821429,0.743025,0.571429,0.856170,0.857143,0.857143,0.708799,0.831149
5,COOS,ResNet50V2,1,0.904762,1.000000,0.714286,0.833333,0.857143,0.53,55,0.809524,0.757664,0.857143,1.553250,0.821429,0.857143,0.757664,0.957862
6,COOS,ResNet50V2,2,0.809524,0.625000,0.833333,0.714286,0.811111,0.49,55,0.869048,0.703979,0.619048,1.165939,0.869048,0.809524,0.703979,0.981975
7,COOS,ResNet50V2,3,0.857143,0.800000,0.666667,0.727273,0.888889,0.34,42,0.785714,0.984917,0.761905,1.054357,0.785714,0.809524,0.982008,1.018176
8,COOS,ResNet50V2,4,0.904762,1.000000,0.666667,0.800000,0.888889,0.52,41,0.880952,0.721783,0.761905,0.981451,0.892857,0.857143,0.721783,0.915267
9,COOS,ResNet50V2,5,0.523810,0.357143,0.833333,0.500000,0.600000,0.10,55,0.833333,0.684757,0.666667,1.509337,0.833333,0.809524,0.684757,0.944988


In [14]:
# Подготовка данных для расчета взвешенных метрик
print("=== ПОДГОТОВКА ДАННЫХ ===")

# Создаем упрощенную версию для основных метрик
df_cv = df_cv_detailed[['suture_type', 'architecture', 'fold', 'accuracy', 'precision', 'recall', 'f1', 'auc']].copy()

print("Средние результаты по типам швов:")
suture_summary = df_cv.groupby('suture_type')[['accuracy', 'precision', 'recall', 'f1', 'auc']].mean().round(4)
print(suture_summary.to_string())


=== ПОДГОТОВКА ДАННЫХ ===
Средние результаты по типам швов:
               accuracy  precision  recall      f1     auc
suture_type                                               
COOS             0.8238     0.7737  0.7399  0.7230  0.8124
ILS              0.9362     0.9415  0.9233  0.9240  0.9512
IOVS_external    0.8987     0.9034  0.9380  0.9186  0.9131
IOVS_internal    0.8967     0.8843  0.9325  0.9049  0.9239


In [15]:
# Расчет взвешенных метрик для каждого типа шва
print("=== РАСЧЕТ ВЗВЕШЕННЫХ МЕТРИК ПО ТИПАМ ШВОВ ===")

# Определяем веса для каждой метрики
weights = {
    'accuracy': 0.10,
    'precision': 0.20,
    'recall': 0.20,
    'f1': 0.35,
    'auc': 0.15
}

print("Веса метрик:")
for metric, weight in weights.items():
    print(f"  {metric}: {weight}")

# Параметр μ для корректировки влияния вариативности
mu = 0.5

# Рассчитываем взвешенную оценку для каждого фолда
df_cv_with_score = df_cv.copy()
df_cv_with_score['weighted_score'] = (
    df_cv_with_score['accuracy'] * weights['accuracy'] +
    df_cv_with_score['precision'] * weights['precision'] +
    df_cv_with_score['recall'] * weights['recall'] +
    df_cv_with_score['f1'] * weights['f1'] +
    df_cv_with_score['auc'] * weights['auc']
)

print(f"\n=== РЕЗУЛЬТАТЫ ВЗВЕШЕННЫХ МЕТРИК ===")

# Создаем сводную таблицу результатов для каждого типа шва
final_results = []

for suture_type in sorted(df_cv_with_score['suture_type'].unique()):
    print(f"\n{'='*50}")
    print(f"ТИП ШВА: {suture_type}")
    print(f"{'='*50}")
    
    suture_data = df_cv_with_score[df_cv_with_score['suture_type'] == suture_type]
    
    # Рассчитываем статистики по архитектурам
    score_stats = suture_data.groupby('architecture')['weighted_score'].agg([
        'mean', 'std', 'min', 'max', 'count'
    ]).reset_index()
    
    # Рассчитываем скорректированную оценку
    score_stats['score_adj'] = score_stats['mean'] - mu * score_stats['std']
    
    # Добавляем средние значения основных метрик
    main_metrics = suture_data.groupby('architecture')[['accuracy', 'precision', 'recall', 'f1', 'auc']].mean().reset_index()
    score_stats = score_stats.merge(main_metrics, on='architecture')
    
    # Переименовываем колонки для ясности
    score_stats.columns = ['architecture', 'score_mean', 'score_std', 'score_min', 'score_max', 'folds_count', 'score_adj', 'accuracy', 'precision', 'recall', 'f1', 'auc']
    
    # Сортируем по скорректированной оценке
    score_stats = score_stats.sort_values('score_adj', ascending=False).round(4)
    
    print(f"\nРейтинг архитектур для {suture_type}:")
    print(score_stats[['architecture', 'score_adj', 'score_mean', 'score_std', 'f1', 'auc']].to_string(index=False))
    
    print(f"\n🏆 ТОП-3 АРХИТЕКТУРЫ ДЛЯ {suture_type}:")
    for i, (idx, row) in enumerate(score_stats.head(3).iterrows()):
        rank = ["🏆", "🥈", "🥉"][i]
        print(f"{rank} {row['architecture']}: Score_adj = {row['score_adj']:.4f} (F1: {row['f1']:.4f}, AUC: {row['auc']:.4f})")
    
    # Сохраняем результаты для общего сравнения
    for idx, row in score_stats.iterrows():
        final_results.append({
            'suture_type': suture_type,
            'architecture': row['architecture'],
            'score_adj': row['score_adj'],
            'score_mean': row['score_mean'],
            'score_std': row['score_std'],
            'f1': row['f1'],
            'auc': row['auc'],
            'accuracy': row['accuracy'],
            'precision': row['precision'],
            'recall': row['recall']
        })

# Создаем общий DataFrame с результатами
df_final_results = pd.DataFrame(final_results)
print(f"\n{'='*80}")
print("ОБЩАЯ СВОДНАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ")
print(f"{'='*80}")
print(df_final_results.round(4).to_string(index=False))


=== РАСЧЕТ ВЗВЕШЕННЫХ МЕТРИК ПО ТИПАМ ШВОВ ===
Веса метрик:
  accuracy: 0.1
  precision: 0.2
  recall: 0.2
  f1: 0.35
  auc: 0.15

=== РЕЗУЛЬТАТЫ ВЗВЕШЕННЫХ МЕТРИК ===

ТИП ШВА: COOS

Рейтинг архитектур для COOS:
    architecture  score_adj  score_mean  score_std     f1    auc
     DenseNet121     0.7874      0.8173     0.0598 0.7881 0.8529
  EfficientNetB0     0.7457      0.7702     0.0489 0.7338 0.8225
MobileNetV3Large     0.7453      0.7669     0.0432 0.7297 0.8286
           VGG16     0.7289      0.7623     0.0669 0.7162 0.7810
        Xception     0.7000      0.7748     0.1498 0.7402 0.8418
      ResNet50V2     0.6921      0.7515     0.1188 0.7150 0.8092
           VGG19     0.6740      0.7209     0.0937 0.6850 0.8111
     InceptionV3     0.6622      0.7163     0.1082 0.6762 0.7519

🏆 ТОП-3 АРХИТЕКТУРЫ ДЛЯ COOS:
🏆 DenseNet121: Score_adj = 0.7874 (F1: 0.7881, AUC: 0.8529)
🥈 EfficientNetB0: Score_adj = 0.7457 (F1: 0.7338, AUC: 0.8225)
🥉 MobileNetV3Large: Score_adj = 0.7453 (F1: 0.72

In [16]:
# Сравнительный анализ между типами швов
print("\n" + "="*80)
print("СРАВНИТЕЛЬНЫЙ АНАЛИЗ МЕЖДУ ТИПАМИ ШВОВ")
print("="*80)

# Лучшие архитектуры для каждого типа шва
print("\n🏆 ЛУЧШИЕ АРХИТЕКТУРЫ ПО ТИПАМ ШВОВ:")
best_by_suture = df_final_results.loc[df_final_results.groupby('suture_type')['score_adj'].idxmax()]
for _, row in best_by_suture.iterrows():
    print(f"  {row['suture_type']:15} → {row['architecture']:15} (Score_adj: {row['score_adj']:.4f})")

# Средние показатели по типам швов
print(f"\n📊 СРЕДНИЕ ПОКАЗАТЕЛИ ПО ТИПАМ ШВОВ:")
suture_means = df_final_results.groupby('suture_type')[['score_adj', 'f1', 'auc', 'accuracy']].mean().round(4)
suture_means = suture_means.sort_values('score_adj', ascending=False)
print(suture_means.to_string())

# Рейтинг архитектур в среднем по всем типам швов
print(f"\n🏅 ОБЩИЙ РЕЙТИНГ АРХИТЕКТУР (среднее по всем типам швов):")
overall_ranking = df_final_results.groupby('architecture')[['score_adj', 'f1', 'auc', 'accuracy']].mean().round(4)
overall_ranking = overall_ranking.sort_values('score_adj', ascending=False)
print(overall_ranking.to_string())

print(f"\n🔝 ТОП-3 АРХИТЕКТУРЫ В СРЕДНЕМ:")
for i, (arch, row) in enumerate(overall_ranking.head(3).iterrows()):
    rank = ["🏆", "🥈", "🥉"][i]
    print(f"{rank} {arch}: Score_adj = {row['score_adj']:.4f} (F1: {row['f1']:.4f}, AUC: {row['auc']:.4f})")



СРАВНИТЕЛЬНЫЙ АНАЛИЗ МЕЖДУ ТИПАМИ ШВОВ

🏆 ЛУЧШИЕ АРХИТЕКТУРЫ ПО ТИПАМ ШВОВ:
  COOS            → DenseNet121     (Score_adj: 0.7874)
  ILS             → ResNet50V2      (Score_adj: 0.9311)
  IOVS_external   → ResNet50V2      (Score_adj: 0.9152)
  IOVS_internal   → ResNet50V2      (Score_adj: 0.9300)

📊 СРЕДНИЕ ПОКАЗАТЕЛИ ПО ТИПАМ ШВОВ:
               score_adj      f1     auc  accuracy
suture_type                                       
ILS               0.9004  0.9240  0.9512    0.9362
IOVS_external     0.8984  0.9186  0.9131    0.8987
IOVS_internal     0.8921  0.9050  0.9239    0.8967
COOS              0.7170  0.7230  0.8124    0.8238

🏅 ОБЩИЙ РЕЙТИНГ АРХИТЕКТУР (среднее по всем типам швов):
                  score_adj      f1     auc  accuracy
architecture                                         
DenseNet121          0.8699  0.8864  0.9076    0.9051
ResNet50V2           0.8671  0.8820  0.9182    0.9005
VGG16                0.8656  0.8748  0.9040    0.9082
MobileNetV3Large     0.8652 

In [17]:
# Итоговая сводная таблица результатов
print("\n" + "="*80)
print("ИТОГОВАЯ СВОДНАЯ ТАБЛИЦА")
print("="*80)

print("\nВзвешенные оценки (Score_adj) по типам швов и архитектурам:")
pivot_table = df_final_results.pivot_table(values='score_adj', index='architecture', columns='suture_type', aggfunc='mean').round(4)
print(pivot_table.to_string())



ИТОГОВАЯ СВОДНАЯ ТАБЛИЦА

Взвешенные оценки (Score_adj) по типам швов и архитектурам:
suture_type         COOS     ILS  IOVS_external  IOVS_internal
architecture                                                  
DenseNet121       0.7874  0.9077         0.8916         0.8929
EfficientNetB0    0.7457  0.9201         0.8822         0.8838
InceptionV3       0.6622  0.8660         0.8759         0.8518
MobileNetV3Large  0.7453  0.9193         0.9112         0.8852
ResNet50V2        0.6921  0.9311         0.9152         0.9300
VGG16             0.7289  0.9109         0.9052         0.9176
VGG19             0.6740  0.8723         0.8925         0.8716
Xception          0.7000  0.8759         0.9134         0.9040
